# Analytical Solution for Groundwater Response to Sea-Level Rise

This notebook provides an analytical solution for groundwater response to sea-level rise (SLR) in a sloping coastal aquifer. The calculations are based on the work of Morgan & Werner (2016), who commented on Chesnaux (2015). The example is from the Pioneer Valley, Australia.

**References:**
- Morgan, L. K. and A. D. Werner (2016). "Comment on “Closed-form analytical solutions for assessing the consequences of sea-level rise on groundwater resources in sloping coastal aquifers”: paper published in Hydrogeology Journal (2015) 23:1399–1413, by R. Chesnaux." Hydrogeology journal 24(5): 1325-1328.
- Werner & Gallagher (2006). "Characterisation of sea-water intrusion in the Pioneer Valley, Australia using hydrochemistry and three-dim."

## 1. Imports and Setup

In [ ]:
import numpy as np
import math
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

## 2. Aquifer and Scenario Parameters

Here we define the key parameters for the aquifer, observation well, and the sea-level rise scenario. These are the default values used for the static plots and the initial state of the interactive widgets.

In [ ]:
# --- Default Aquifer Parameters ---
DEFAULT_K = 100         # Hydraulic conductivity (m/d)
DEFAULT_Z0 = 25         # Aquifer thickness (m)
DEFAULT_RHO_F = 1000    # Density of freshwater (kg/m^3)
DEFAULT_RHO_S = 1025    # Density of saltwater (kg/m^3)
DEFAULT_W = 0.11/365.25 # Net recharge (m/d)

# --- Default Observation Well Data ---
DEFAULT_HB = 2          # Hydraulic head at observation well (m above MSL)
DEFAULT_XB = 2000       # Distance of observation well from the coast (m)

# --- Default Sea Level Rise Scenario ---
DEFAULT_DELTA_Z = 1.0     # Sea Level Rise (m)

## 3. Core Calculation and Analysis Functions

These functions perform the core hydrogeological calculations and analyze the two main scenarios: flux-controlled and head-controlled.

In [ ]:
def calculate_delt(rho_f, rho_s):
    return (rho_s - rho_f) / rho_f

def calculate_coastal_flow(hb, xb, z0, k, w, delt):
    qb = ((((hb + z0)**2 - ((1 + delt) * z0**2)) * k) - (w * xb**2)) / (2 * xb)
    return qb + (w * xb)

def calculate_groundwater_divide(q0, w):
    if w == 0: return np.inf
    return q0 / w

def calculate_mixed_convection_ratio(k, delt, z0, w, xn):
    if w <= 0 or xn <= 0 or not np.isfinite(xn): return np.inf
    return k * delt * (1 + delt) * z0**2 / (w * xn**2)

def calculate_interface_toe_position(xn, m):
    if 1 - m < 0 or not np.isfinite(m): return xn
    return xn * (1 - np.sqrt(1 - m))

def calculate_head_at_x(x, q0, w, k, z0, delt):
    inside_sqrt = (2 * q0 * x - w * x**2) / k + (1 + delt) * z0**2
    with np.errstate(invalid='ignore'):
        head = np.sqrt(inside_sqrt) - z0
    return head

def analyze_flux_controlled(q0, w, k, z0, delt, delta_z):
    z0_post_slr = z0 + delta_z
    xn = calculate_groundwater_divide(q0, w)
    m_post_slr = calculate_mixed_convection_ratio(k, delt, z0_post_slr, w, xn)
    xt_post_slr = calculate_interface_toe_position(xn, m_post_slr)
    return {'q0_post_slr': q0, 'xn_post_slr': xn, 'xt_post_slr': xt_post_slr}

def analyze_head_controlled(hb, xb, w, k, z0, delt, delta_z):
    z0_post_slr = z0 + delta_z
    qb_post_slr = ((((hb + z0)**2 - ((1 + delt) * z0_post_slr**2)) * k) - (w * xb**2)) / (2 * xb)
    q0_post_slr = qb_post_slr + (w * xb)
    xn_post_slr = calculate_groundwater_divide(q0_post_slr, w)
    m_post_slr = calculate_mixed_convection_ratio(k, delt, z0_post_slr, w, xn_post_slr)
    xt_post_slr = calculate_interface_toe_position(xn_post_slr, m_post_slr)
    return {'q0_post_slr': q0_post_slr, 'xn_post_slr': xn_post_slr, 'xt_post_slr': xt_post_slr}

## 4. Static Visualization of Scenarios

This section shows a static comparison of the Pre-SLR conditions versus the two Post-SLR scenarios: flux-controlled and head-controlled. This highlights the different impacts each boundary condition has on the saltwater intrusion.

In [ ]:
def plot_comparison_scenario(ax, title, x, z0, delta_z, pre_data, post_data):
    """Helper function to plot a comparison of pre- and post-SLR scenarios."""
    h_pre, xt_pre = pre_data['h'], pre_data['xt']
    h_post, xt_post = post_data['h'], post_data['xt']
    
    ax.plot(x, h_pre, 'b-', label='Head (Pre-SLR)')
    ax.fill_between(x, -z0, h_pre, color='lightblue', alpha=0.5)
    ax.axvline(xt_pre, color='b', linestyle='--', label=f'Toe Pre-SLR: {xt_pre:.2f} m')

    # The calculated post-SLR head (h_post) is relative to the new sea level.
    # We add delta_z to plot it relative to the original MSL datum.
    ax.plot(x, h_post + delta_z, 'r-', label='Head (Post-SLR)')
    ax.fill_between(x, -(z0), h_post + delta_z, where=x>xt_post, color='lightcoral', alpha=0.4)

    ax.axhline(y=0, color='blue', linestyle=':', label='Original MSL')
    ax.axhline(y=delta_z, color='red', linestyle=':', label=f'New MSL (+{delta_z}m)')
    ax.axhline(y=-z0, color='k', linestyle='-', label='Aquifer Base')

    ax.set_title(title)
    ax.set_xlabel('Distance from Coast (m)')
    ax.set_ylabel('Elevation (m)')
    ax.legend(loc='upper right')
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.set_xlim(0, max(x))
    max_h = max(np.nanmax(h_pre), np.nanmax(h_post + delta_z)) if np.any(np.isfinite(h_pre)) else 10
    ax.set_ylim(-z0 - 10, max_h + 5)
    
# --- Run Calculations for Static Plots ---
delt = calculate_delt(DEFAULT_RHO_F, DEFAULT_RHO_S)
z0_post_slr = DEFAULT_Z0 + DEFAULT_DELTA_Z

# Pre-SLR
q0_pre = calculate_coastal_flow(DEFAULT_HB, DEFAULT_XB, DEFAULT_Z0, DEFAULT_K, DEFAULT_W, delt)
xn_pre = calculate_groundwater_divide(q0_pre, DEFAULT_W)
xt_pre = calculate_interface_toe_position(xn_pre, calculate_mixed_convection_ratio(DEFAULT_K, delt, DEFAULT_Z0, DEFAULT_W, xn_pre))
x_range = np.linspace(0, xn_pre * 1.1, 500)
h_pre = calculate_head_at_x(x_range, q0_pre, DEFAULT_W, DEFAULT_K, DEFAULT_Z0, delt)
pre_data = {'h': h_pre, 'xt': xt_pre}

# Post-SLR Flux-Controlled
flux_res = analyze_flux_controlled(q0_pre, DEFAULT_W, DEFAULT_K, DEFAULT_Z0, delt, DEFAULT_DELTA_Z)
h_flux_prime = calculate_head_at_x(x_range, flux_res['q0_post_slr'], DEFAULT_W, DEFAULT_K, z0_post_slr, delt)
flux_data = {'h': h_flux_prime, 'xt': flux_res['xt_post_slr']}

# Post-SLR Head-Controlled
head_res = analyze_head_controlled(DEFAULT_HB, DEFAULT_XB, DEFAULT_W, DEFAULT_K, DEFAULT_Z0, delt, DEFAULT_DELTA_Z)
h_head_prime = calculate_head_at_x(x_range, head_res['q0_post_slr'], DEFAULT_W, DEFAULT_K, z0_post_slr, delt)
head_data = {'h': h_head_prime, 'xt': head_res['xt_post_slr']}

# --- Create Plots ---
fig, axes = plt.subplots(2, 1, figsize=(12, 14))
plot_comparison_scenario(axes[0], 'Comparison: Pre-SLR vs. Post-SLR (Flux-Controlled)', x_range, DEFAULT_Z0, DEFAULT_DELTA_Z, pre_data, flux_data)
plot_comparison_scenario(axes[1], 'Comparison: Pre-SLR vs. Post-SLR (Head-Controlled)', x_range, DEFAULT_Z0, DEFAULT_DELTA_Z, pre_data, head_data)
plt.tight_layout()
plt.show()

## 5. Interactive Simulation of Scenarios

This section provides an interactive tool to explore both the **Flux-Controlled** and **Head-Controlled** scenarios. Use the sliders to change the input parameters and observe the real-time effect on both models simultaneously.

In [ ]:
def plot_interactive_scenarios(k, z0, w, delta_z, hb, xb):
    """Function to run simulation and plot results for both scenarios interactively."""
    delt = calculate_delt(DEFAULT_RHO_F, DEFAULT_RHO_S)
    z0_post_slr = z0 + delta_z
    
    # Pre-SLR calculations (base for both scenarios)
    q0_pre = calculate_coastal_flow(hb, xb, z0, k, w, delt)
    xn_pre = calculate_groundwater_divide(q0_pre, w)
    m_pre = calculate_mixed_convection_ratio(k, delt, z0, w, xn_pre)
    xt_pre = calculate_interface_toe_position(xn_pre, m_pre)

    # --- Scenario-specific calculations ---
    flux_res = analyze_flux_controlled(q0_pre, w, k, z0, delt, delta_z)
    head_res = analyze_head_controlled(hb, xb, w, k, z0, delt, delta_z)

    # Determine plot range
    max_xn = max(xn_pre, flux_res['xn_post_slr'], head_res['xn_post_slr'])
    if not np.isfinite(max_xn): max_xn = 5000
    x = np.linspace(0, max_xn * 1.1, 500)
    
    # Calculate heads (h_prime is relative to new MSL)
    h_pre = calculate_head_at_x(x, q0_pre, w, k, z0, delt)
    h_flux_prime = calculate_head_at_x(x, flux_res['q0_post_slr'], w, k, z0_post_slr, delt)
    h_head_prime = calculate_head_at_x(x, head_res['q0_post_slr'], w, k, z0_post_slr, delt)
    
    pre_data = {'h': h_pre, 'xt': xt_pre}
    flux_data = {'h': h_flux_prime, 'xt': flux_res['xt_post_slr']}
    head_data = {'h': h_head_prime, 'xt': head_res['xt_post_slr']}
    
    # --- Create Plots ---
    fig, axes = plt.subplots(1, 2, figsize=(20, 8), sharey=True)
    plot_comparison_scenario(axes[0], 'Interactive: Flux-Controlled Scenario', x, z0, delta_z, pre_data, flux_data)
    plot_comparison_scenario(axes[1], 'Interactive: Head-Controlled Scenario', x, z0, delta_z, pre_data, head_data)
    plt.tight_layout()
    plt.show()

# Create and display widgets
interactive_plot = widgets.interactive(plot_interactive_scenarios, 
                                       k=widgets.FloatSlider(value=DEFAULT_K, min=1, max=500, step=1, description='K (m/d):'),
                                       z0=widgets.FloatSlider(value=DEFAULT_Z0, min=5, max=50, step=1, description='z0 (m):'),
                                       w=widgets.FloatSlider(value=DEFAULT_W, min=0, max=0.001, step=0.00005, description='Recharge (m/d):', readout_format='.5f'),
                                       delta_z=widgets.FloatSlider(value=DEFAULT_DELTA_Z, min=0, max=5, step=0.1, description='SLR (m):'),
                                       hb=widgets.FloatSlider(value=DEFAULT_HB, min=0, max=10, step=0.5, description='Inland Head (m):'),
                                       xb=widgets.FloatSlider(value=DEFAULT_XB, min=500, max=5000, step=100, description='Inland Dist. (m):'))
display(interactive_plot)